In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/robinaz/crypto/crypto.csv


# **Python Practice: Building Time Series Models (MA, AR and ARMA)**

In Week 8 we checked Bitcoin (BTC) for two things:
1. **Stationarity:** plot + ADF test.
2. **Autocorrelation:** ACF/PACF plots + Ljung–Box test.

We found that BTC **prices** are non-stationary and strongly autocorrelated. BTC **returns** are stationary, with almost no autocorrelation (Ljung–Box p = 0.055).

This week we **build time series models** for BTC returns and test, model by model, whether the past helps to predict today. We start with **MA(1)**, then **AR(1)**, then **ARMA(1,1)**.

## **Step 1: Import the Libraries**

Same as Week 8, plus one new tool: `ARIMA`. We use it to estimate MA, AR and ARMA models.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings("ignore")

## **Step 2: Upload the Data and Compute Returns**


As in Week 8, we compute daily log returns in %. We model **returns, not prices**, because returns are stationary.

In [13]:
df = pd.read_csv("/kaggle/input/datasets/robinaz/crypto/crypto.csv")
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")
df = df.set_index("Date")

df.head()

,BTC
Date,
2015-01-01,201.7391
2015-01-02,205.5674
2015-01-03,183.4140
2015-01-04,172.8799
2015-01-05,179.9004


# Returns Calculation

In [14]:
returns = np.log(df["BTC"]).diff().dropna() * 100
returns.head()

Date
2015-01-02     1.879868
2015-01-03   -11.402807
2015-01-04    -5.914876
2015-01-05     3.980623
2015-01-06     4.975838
Name: BTC, dtype: float64

## **How to Write a Model in Python**

Every model is written as `order=(p, d, q)`:

- **p** = number of past **returns** (AR part)
- **d** = number of differences. We use **0**, because returns are already stationary
- **q** = number of past **shocks** (MA part)

| Model | Order |
|---|---|
| MA(1) | (0, 0, 1) |
| AR(1) | (1, 0, 0) |
| ARMA(1,1) | (1, 0, 1) |

## **Step 3: Estimate an MA(1) Model**

$r_t = \mu + u_t + \theta_1 u_{t-1}$

Today's return = average return + today's shock (news) + a share θ₁ of yesterday's shock.

In [4]:
ma1 = ARIMA(returns, order=(0, 0, 1)).fit()
print(ma1.summary())

                               SARIMAX Results                                
Dep. Variable:                    BTC   No. Observations:                 4017
Model:                 ARIMA(0, 0, 1)   Log Likelihood              -10767.120
Date:                Sat, 14 Nov 2026   AIC                          21540.240
Time:                        20:40:09   BIC                          21559.135
Sample:                    01-02-2015   HQIC                         21546.937
                         - 12-31-2025                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.1438      0.056      2.578      0.010       0.034       0.253
ma.L1         -0.0221      0.009     -2.381      0.017      -0.040      -0.004
sigma2        12.4647      0.111    112.336      0.0

**How to read the table:**

| Row | Meaning |
|---|---|
| `const` | μ, the average daily return (%) |
| `ma.L1` | θ₁, the effect of yesterday's shock |
| `sigma2` | the variance of the shocks |
| `P>|z|` | p-value. If p < 0.05, the coefficient is significant |

In [5]:
print("Average daily return (const):", round(ma1.params["const"], 4), "%")
print("theta_1 (ma.L1):", round(ma1.params["ma.L1"], 4))
print("p-value of theta_1:", round(ma1.pvalues["ma.L1"], 4))

Average daily return (const): 0.1438 %
theta_1 (ma.L1): -0.0221
p-value of theta_1: 0.0173


**Interpretation:**
- **Average daily return (`const`) = 0.1438%.** On average, BTC rose by about 0.14% per day over the sample. That is roughly three times the average daily return of the S&P 500.
- **θ₁ = −0.0221.** This is **very small and negative**. If yesterday's shock (news) was +10%, about 0.22% of it is **reversed** today (−0.0221 × 10% = −0.221%).
- **p-value = 0.0173**, which is below 0.05, so θ₁ is **statistically significant**.

So the effect is **real but tiny**. Yesterday's news has a small, significant effect on today's return, but far too small to be useful for trading. With about 4,000 daily observations, even a very small effect can be significant.

This fits Week 8: the Ljung–Box test found almost no autocorrelation (p = 0.055), and here the MA(1) picks up a tiny bit at lag 1.

> ## **Step 4: Estimate an AR(1) Model**

$r_t = \mu + \phi_1 r_{t-1} + u_t$

Today's return depends on **yesterday's return**, plus a shock.
MA uses past **shocks**; AR uses past **returns**.

In [6]:
ar1 = ARIMA(returns, order=(1, 0, 0)).fit()
print(ar1.summary())

                               SARIMAX Results                                
Dep. Variable:                    BTC   No. Observations:                 4017
Model:                 ARIMA(1, 0, 0)   Log Likelihood              -10767.096
Date:                Sat, 14 Nov 2026   AIC                          21540.191
Time:                        20:42:29   BIC                          21559.086
Sample:                    01-02-2015   HQIC                         21546.887
                         - 12-31-2025                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.1438      0.056      2.579      0.010       0.035       0.253
ar.L1         -0.0227      0.009     -2.428      0.015      -0.041      -0.004
sigma2        12.4643      0.111    112.336      0.0

In [7]:
phi = ar1.params["ar.L1"]
print("phi_1 (ar.L1):", round(phi, 4))
print("p-value of phi_1:", round(ar1.pvalues["ar.L1"], 4))
print("|phi_1| < 1 -> stationary?", abs(phi) < 1)

phi_1 (ar.L1): -0.0227
p-value of phi_1: 0.0152
|phi_1| < 1 -> stationary? True


**Interpretation:**
- **φ₁ = −0.0227.** This is **very small and negative**. If yesterday's BTC return was 10% above average, today's return is expected to be about 0.23% **below** average (−0.0227 × 10% = −0.227%). This is a tiny "bounce-back" effect.
- **p-value = 0.0152**, which is below 0.05, so φ₁ is **statistically significant**. But it is far too small to be useful for trading.
- **|φ₁| = 0.0227 < 1**, so the model is **stationary**. A shock fades away almost immediately: after 2 days, only 0.0227² ≈ 0.0005 of it is left.

The AR(1) gives almost the **same** answer as the MA(1) (φ₁ = −0.0227 vs θ₁ = −0.0221). When the coefficient is this small, past **returns** and past **shocks** tell the same story: a tiny, short-lived pattern at lag 1.

## **Step 5: Estimate an ARMA(1,1) Model**

$r_t = \mu + \phi_1 r_{t-1} + u_t + \theta_1 u_{t-1}$

ARMA combines both: yesterday's **return** and yesterday's **shock**.

In [9]:
arma11 = ARIMA(returns, order=(1, 0, 1)).fit()
print(arma11.summary())

                               SARIMAX Results                                
Dep. Variable:                    BTC   No. Observations:                 4017
Model:                 ARIMA(1, 0, 1)   Log Likelihood              -10766.956
Date:                Sat, 14 Nov 2026   AIC                          21541.911
Time:                        20:43:47   BIC                          21567.105
Sample:                    01-02-2015   HQIC                         21550.840
                         - 12-31-2025                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.1438      0.056      2.564      0.010       0.034       0.254
ar.L1         -0.2705      0.477     -0.567      0.571      -1.206       0.665
ma.L1          0.2473      0.478      0.517      0.6

In [10]:
print("phi_1 (ar.L1):", round(arma11.params["ar.L1"], 4), " p-value:", round(arma11.pvalues["ar.L1"], 4))
print("theta_1 (ma.L1):", round(arma11.params["ma.L1"], 4), " p-value:", round(arma11.pvalues["ma.L1"], 4))

phi_1 (ar.L1): -0.2705  p-value: 0.5708
theta_1 (ma.L1): 0.2473  p-value: 0.605


**Interpretation:**
- **φ₁ = −0.2705 (p = 0.5708)** and **θ₁ = 0.2473 (p = 0.605)**. Both p-values are well **above 0.05**, so **neither** coefficient is significant.
- The two coefficients look bigger than in the MA(1) and AR(1), but they have **opposite signs and almost the same size**. They **cancel each other out**: −0.2705 + 0.2473 = −0.0232, which is almost exactly the φ₁ and θ₁ we found before (about −0.022).
- So the ARMA(1,1) does not find anything new. It only splits the same tiny lag 1 effect into two parts that cancel. This is a clear sign the model is **bigger than it needs to be**.

The ARMA(1,1) is **not a good model** for BTC returns. A simpler model, MA(1) or AR(1), captures the same pattern with fewer coefficients. Next week we confirm this with AIC and SBIC.

## **Step 6: Compare the Three Models**

In [12]:
comparison = pd.DataFrame({
    "MA(1)": ma1.params,
    "AR(1)": ar1.params,
    "ARMA(1,1)": arma11.params,
}).round(4)
comparison = comparison.reindex(["const", "ar.L1", "ma.L1", "sigma2"]).fillna("–")
comparison

,MA(1),AR(1),"ARMA(1,1)"
const,0.1438,0.1438,0.1438
ar.L1,–,-0.0227,-0.2705
ma.L1,-0.0221,–,0.2473
sigma2,12.4647,12.4643,12.4634


**Interpretation:**
- **NaN** means "not in this model". The MA(1) has no `ar.L1`, and the AR(1) has no `ma.L1`.
- **`const` = 0.1438% in all three models.** The average daily return does not change, whichever model we use.
- **`sigma2` ≈ 12.46 in all three models.** This is the variance of the shocks. Its square root (√12.46 ≈ 3.53%) is the typical size of a daily BTC shock. Adding AR or MA terms hardly reduces it (12.4647 → 12.4634), so these models explain almost **nothing** of BTC's daily moves.
- **MA(1)** (θ₁ = −0.0221) and **AR(1)** (φ₁ = −0.0227) find the same **tiny** negative effect at lag 1.
- **ARMA(1,1)**: φ₁ = −0.2705 and θ₁ = +0.2473 **cancel each other out**, and neither is significant.

The past tells us **very little** about today's BTC return. This is a **finding, not a failure**: it fits the efficient markets idea that the average return is hard to predict.

## **Conclusion**

| Model | Coefficient | p-value | Significant? |
|---|---|---|---|
| MA(1) | θ₁ = −0.0221 | 0.017 | Yes, but tiny |
| AR(1) | φ₁ = −0.0227 | 0.015 | Yes, but tiny |
| ARMA(1,1) | φ₁ = −0.2705, θ₁ = 0.2473 | 0.57, 0.61 | No, they cancel out |

- BTC's average daily return is about **0.14%**.
- MA(1) and AR(1) find the same **tiny** negative effect at lag 1: too small to be useful.
- ARMA(1,1) is **too big**: its two terms cancel out.
- The **average** BTC return is **hard to predict** from its own past.

**Next week:** we use AIC and SBIC to choose the best model, check it, and then model BTC's **risk** (volatility) with ARCH and GARCH.